### Import Libs :

In [32]:
import pandas as pd
from sklearn.cluster import KMeans
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import numpy as np
import matplotlib as plt
import seaborn as sns


### Read and explore Data :

In [2]:
df = pd.read_csv('../data/questions.csv')

In [3]:
df.head()

,Question,Answer Status
0,Does the PDF explain what is the largest ocean...,Not Exist
1,How many moons does Jupiter have?,Not Exist
2,Tell me what is the plot of the movie 'incepti...,Not Exist
3,How do you tie a tie?,Not Exist
4,Is there information about who is the lead sin...,Not Exist


In [4]:
df.groupby('Answer Status').count()


,Question
Answer Status,
Exist,500
Not Exist,500


### embedding :

In [5]:
loader = CSVLoader(
    file_path='../data/questions.csv',
    source_column='Question', 
)

documents = loader.load()

model_name = "BAAI/bge-m3"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

vectorstore = FAISS.from_documents(documents, embeddings)
vectorstore.save_local("../faiss_store")

/media/rachid/d70e3dc6-74e7-4c87-96bc-e4c3689c979a/lmobrmij/Projects/Rag_It_Assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
index = vectorstore.index
dimension = index.d
vectors = np.array([vectorstore.index.reconstruct(i) for i in range(index.ntotal)])
print(dimension)
print(index.ntotal)
print(vectors.shape)

1024
1000
(1000, 1024)


In [21]:
# Training model
kmeans = KMeans(n_clusters=2)
X = kmeans.fit(vectors)

In [24]:
# Getting cluster labels and centroids
labels = kmeans.labels_
centroids = kmeans.cluster_centers_

# Printing results
print(f"Number of labels: {len(labels)}")
print(f"Labels : {labels}")
print(f"Get label of question 304 : {labels[305]}")
print(centroids)

Number of labels: 1000
Labels : [0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 1 1 0 0 0 1 0 0 1 0 0 0 1 1 0 1 0 0 1
 1 0 1 0 1 1 1 1 1 0 1 1 0 1 0 1 0 0 1 1 1 0 0 1 0 0 1 0 0 0 1 1 1 1 1 1 1
 0 0 1 1 1 0 1 1 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0 0 0 1 1 0 1 0 0 0 1
 1 0 1 0 0 0 1 0 1 0 1 1 0 1 1 0 1 1 1 1 1 1 0 0 1 1 1 1 1 1 1 0 0 0 1 1 0
 0 1 1 0 0 0 1 1 1 0 1 0 0 0 0 0 0 1 1 0 0 1 1 1 1 1 0 0 0 1 0 1 0 1 0 0 1
 0 0 0 0 1 1 0 1 1 0 0 0 0 1 1 1 1 0 0 1 0 0 0 1 1 1 0 0 1 0 1 0 1 1 1 0 0
 0 1 1 0 1 0 1 0 0 1 1 0 1 1 0 0 0 0 0 0 1 0 1 1 1 0 1 0 0 0 0 0 1 1 0 1 1
 0 0 0 0 1 1 1 0 1 1 0 1 0 0 0 1 1 0 0 1 0 0 1 0 1 1 0 1 1 1 1 1 0 1 0 1 1
 1 1 0 0 1 1 0 0 1 0 1 1 1 0 1 1 1 0 0 0 1 0 1 0 1 0 1 1 1 1 0 1 0 1 0 0 1
 0 1 0 1 0 1 0 0 1 1 0 1 0 1 1 0 1 1 0 0 1 0 1 0 0 1 1 0 0 1 1 1 1 1 1 1 1
 0 1 1 0 1 0 1 0 0 1 0 0 1 1 0 0 0 1 1 0 1 1 0 0 0 0 0 1 1 1 0 1 1 0 0 0 1
 1 0 1 1 0 0 1 0 1 0 0 0 0 0 0 1 1 1 1 1 0 1 1 1 0 0 1 0 0 0 0 0 1 1 1 1 1
 1 1 1 0 1 1 0 0 1 0 0 0 0 0 1 0 1 0 0 1 0 1 1 1 0 1 1 1 0 1 0 0 0 0

In [ ]:
import numpy as np

def predict_label(question, clustering_model, encoder):
    vector = encoder.embed_query(question)

    vector = np.array(vector, dtype=np.double).reshape(1, -1)

    label = clustering_model.predict(vector)[0]
    return label
    
question = "How can I reset my password?"
label = predict_label(question, kmeans, embeddings)

print(f"Question: {question} -> Label: {label}")


ValueError: Buffer dtype mismatch, expected 'const double' but got 'float'